# BTC keyframe augmentation tích hợp Quang–Trọng — smoke → DEV pilot

Notebook này dùng **engine thực thi của Trọng** và **contract chấp nhận của Quang**
cho phương án
`BTC-first, audit-gated dynamic-density delta` trong
`docs/research/keyframe-augmentation/implementation-runbook.md`.

| Chế độ | Phạm vi | Kết quả |
|---|---|---|
| `SMOKE` | Đúng một video DEV | Kiểm tra kỹ thuật end-to-end và tín hiệu hiệu quả, sinh `smoke-report.json` vào local |
| `PILOT` | 51 DEV + 24 video mẫu phân tầng | Gate 0 → TransNet/audit → oracle → B1–B4 → DEV A/B → freeze |

Mặc định là **SMOKE + local output**. Notebook không có chế độ chạy 873 video
và không mở held-out trước khi có `freeze.json`. Ở SMOKE, Modal (nếu bật) chỉ
là nguồn **đọc** manifest/map/keyframe BTC; mọi augmentation artifact nằm dưới
`/kaggle/working` và không có lệnh ghi vào Volume của nhóm.

Các invariant không được nới lỏng:

- `frame_idx` và `pts_time` lấy trực tiếp từ decoder/CSV; không suy ra từ ordinal;
- identity là `build_keyframe_id(video_id, frame_idx)`;
- BTC base bundle và SQLite không bao giờ bị ghi đè;
- B1–B4 phải cùng hard delta budget; underfill làm A/B chính bị BLOCKED;
- B4 là random control cùng ngân sách sau dedup;
- O1-T không được dùng để suy ra hết headroom; thiếu O1-S/O1-R thật phải BLOCKED;
- lỗi dữ liệu/model là `FAILED`, phương án không đạt điểm là `NO_GO`.

## 0 — Chỉ sửa cell cấu hình này

In [ ]:
# ── MODE / SAFETY ──────────────────────────────────────────────────────────
import re

RUN_MODE = "SMOKE"                 # "SMOKE" hoặc "PILOT"
MEMBER = "quang"                   # owner provenance, không đổi thuật toán
WORKER = "quang1"                  # shard hợp lệ của MEMBER
WORKER_ASSIGNMENTS = {
    "triet": ("triet1", "triet2"),
    "trong": ("trong1", "trong2"),
    "quang": ("quang1", "quang2", "quang3", "quang4"),
    "bao": ("bao1", "bao2"),
    "dat": ("dat1", "dat2"),
}
EXPERIMENT_ID = "btc_aug_cov_v1"
CONFIRM_PILOT = ""                 # PILOT yêu cầu đúng EXPERIMENT_ID
OUTPUT_TRANSPORT = "local"         # SMOKE bắt buộc là local
ALLOW_MODAL_WRITES = False          # chỉ Lead mới mở khi đã review PILOT

# Clone đúng commit đã review. Branch chỉ phù hợp để smoke; PILOT sẽ từ chối
# nếu REPO_REF không phải full 40-hex commit.
REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang-trong/keyframe-integration"

VOLUME = "keyframes-btc"
VOLUME_VERSION = 2
PARENT_BUNDLE = "aic_btc_b1_v1"
SMOKE_VIDEO_ID = "L21_V007"        # hoặc "AUTO" để chọn DEV có gap lớn nhất trong input
PARENT_MANIFEST_SOURCE = "modal"   # "modal" hoặc đường dẫn manifest attach sẵn
READ_SMOKE_BTC_FROM_MODAL = True    # chỉ fetch map/keyframe của đúng SMOKE_VIDEO_ID

# Grid mặc định dùng cho O1 và smoke; PILOT sẽ quét toàn bộ gap × k_max sau
# khi threshold được calibration trên DEV.
DEFAULT_THRESHOLD = 0.5
DEFAULT_TARGET_GAP = 6.0
DEFAULT_K_MAX = 3
FEATURE_STAGES = "n4_siglip,n5_ocr_easyocr"
LATENCY_REPEATS = 1                 # smoke nhanh; PILOT nên tăng lên >= 3

# Kaggle input được dò tự động. TransNet dùng đúng TensorFlow SavedModel đã
# chạy trong shotdetect_dake.ipynb; code inference clone từ official repo.
INPUT_ROOT = "/kaggle/input"
SAVED_MODEL_GLOB = "AUTO"          # hoặc đường dẫn .../saved_model.pb
SMOKE_CHECKPOINT_SHA256 = "AUTO"    # smoke ghi nhận SHA thật; PILOT vẫn yêu cầu SHA đã pin trong config
TRANSNET_REPO_URL = "https://github.com/soCzech/TransNetV2.git"
TRANSNET_REPO_REF = "master"        # SMOKE ghi nhận commit thực; PILOT yêu cầu full SHA
BASE_FLAT_GLOB = "AUTO"
BASE_DB_GLOB = "AUTO"
LABELS_PATH = "augmentation_labels/transnet_dev.jsonl"

if RUN_MODE not in {"SMOKE", "PILOT"}:
    raise ValueError("RUN_MODE must be SMOKE or PILOT")
if MEMBER not in WORKER_ASSIGNMENTS or WORKER not in WORKER_ASSIGNMENTS[MEMBER]:
    raise ValueError(f"Worker {WORKER!r} không thuộc MEMBER={MEMBER!r}")
if OUTPUT_TRANSPORT not in {"local", "modal"}:
    raise ValueError("OUTPUT_TRANSPORT must be local or modal")
if RUN_MODE == "SMOKE" and OUTPUT_TRANSPORT != "local":
    raise RuntimeError("SMOKE bắt buộc OUTPUT_TRANSPORT='local'; không ghi Modal")
if OUTPUT_TRANSPORT == "modal" and not ALLOW_MODAL_WRITES:
    raise RuntimeError("Ghi Modal đang khóa; cần Lead review và ALLOW_MODAL_WRITES=True")
if RUN_MODE == "PILOT" and CONFIRM_PILOT != EXPERIMENT_ID:
    raise RuntimeError(
        "PILOT bị khóa an toàn. Đặt CONFIRM_PILOT đúng bằng EXPERIMENT_ID "
        "sau khi đã review input/checkpoint."
    )
if RUN_MODE == "PILOT" and not re.fullmatch(r"[0-9a-f]{40}", TRANSNET_REPO_REF):
    raise RuntimeError("PILOT yêu cầu TRANSNET_REPO_REF là full 40-hex commit SHA")

## 1 — Kaggle runtime, private repo và dependency fingerprint

In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path


def kaggle_secret(name: str, *, required: bool = True) -> str:
    """Read one Kaggle Secret without ever printing its value."""
    from kaggle_secrets import UserSecretsClient

    try:
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = ""
    if required and not value:
        raise RuntimeError(f"Thiếu Kaggle Secret {name}")
    return value or ""


github_token = kaggle_secret("GITHUB_TOKEN")


def _redact_text(value: object, secrets: tuple[str, ...]) -> str:
    text = str(value)
    for secret in secrets:
        if secret:
            text = text.replace(secret, "<redacted>")
    return text


def _run_private_git(
    command: list[str], *, env: dict[str, str], cwd: Path | None = None
) -> None:
    """Run authenticated Git without allowing a secret into argv or traceback."""
    try:
        subprocess.run(
            command,
            cwd=cwd,
            env=env,
            check=True,
            capture_output=True,
            text=True,
        )
    except subprocess.CalledProcessError as error:
        safe_command = [_redact_text(part, (github_token,)) for part in error.cmd]
        safe_stdout = _redact_text(error.stdout or "", (github_token,)).strip()
        safe_stderr = _redact_text(error.stderr or "", (github_token,)).strip()
        details = safe_stderr or safe_stdout or "no Git output"
        raise RuntimeError(
            f"Git command failed ({error.returncode}): {safe_command!r}; {details}"
        ) from None


def _private_git_environment() -> tuple[dict[str, str], Path]:
    handle, askpass_name = tempfile.mkstemp(prefix="mfe-git-askpass-", suffix=".sh")
    os.close(handle)
    askpass = Path(askpass_name)
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in\n'
        '*Username*) printf "%s\\n" "x-access-token" ;;\n'
        '*Password*) printf "%s\\n" "$MFE_GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["MFE_GITHUB_TOKEN"] = github_token
    return env, askpass


needs_modal_read = PARENT_MANIFEST_SOURCE == "modal" or READ_SMOKE_BTC_FROM_MODAL
needs_modal = needs_modal_read or OUTPUT_TRANSPORT == "modal"
if needs_modal:
    os.environ["MODAL_TOKEN_ID"] = kaggle_secret("MODAL_TOKEN_ID")
    os.environ["MODAL_TOKEN_SECRET"] = kaggle_secret("MODAL_TOKEN_SECRET")

REPO = Path("/kaggle/working/mfe")
attached_repo = Path("/kaggle/input/mfe-repo")
if not REPO.is_dir():
    # Gate 0 records a committed source revision.  Only use an attached repo
    # when its Git metadata is present; a Kaggle dataset containing source files
    # alone would otherwise make the lock unverifiable and fail later.
    use_attached = (
        attached_repo.is_dir()
        and (attached_repo / ".git").is_dir()
        and not re.fullmatch(r"[0-9a-f]{40}", REPO_REF)
    )
    if use_attached:
        def _ignore(_directory, names):
            return {name for name in names if name in {"data", "stores"}}

        shutil.copytree(attached_repo, REPO, ignore=_ignore)
    else:
        git_env, askpass_path = _private_git_environment()
        try:
            if re.fullmatch(r"[0-9a-f]{40}", REPO_REF):
                _run_private_git(
                    ["git", "clone", "--filter=blob:none", REPO_URL, str(REPO)],
                    env=git_env,
                )
                _run_private_git(
                    ["git", "fetch", "--depth", "1", "origin", REPO_REF],
                    cwd=REPO,
                    env=git_env,
                )
                _run_private_git(
                    ["git", "checkout", "--detach", REPO_REF],
                    cwd=REPO,
                    env=git_env,
                )
            else:
                _run_private_git(
                    ["git", "clone", "--depth", "1", "-b", REPO_REF, REPO_URL, str(REPO)],
                    env=git_env,
                )
        finally:
            git_env.pop("MFE_GITHUB_TOKEN", None)
            askpass_path.unlink(missing_ok=True)
sys.path.insert(0, str(REPO / "src"))

if (REPO / ".git").is_dir():
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
    ).strip()
else:
    git_commit = "attached-repo-without-git"
if RUN_MODE == "PILOT" and not re.fullmatch(r"[0-9a-f]{40}", REPO_REF):
    raise RuntimeError("PILOT yêu cầu REPO_REF là full 40-hex commit SHA")
if RUN_MODE == "PILOT" and re.fullmatch(r"[0-9a-f]{40}", REPO_REF) and git_commit != REPO_REF:
    raise RuntimeError(f"repo commit mismatch: requested {REPO_REF}, got {git_commit}")

requirements = REPO / "notebooks" / "requirements-kaggle-btc-augmentation.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
    check=True,
)

# Clone only official inference code. We explicitly skip Git-LFS weights because
# the reviewed SavedModel is attached read-only as a Kaggle Model/Dataset.
TRANSNET_REPO = Path("/kaggle/working/TransNetV2")
if not TRANSNET_REPO.is_dir():
    transnet_env = os.environ.copy()
    transnet_env["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(
        ["git", "clone", "--filter=blob:none", TRANSNET_REPO_URL, str(TRANSNET_REPO)],
        check=True,
        capture_output=True,
        env=transnet_env,
    )
    if re.fullmatch(r"[0-9a-f]{40}", TRANSNET_REPO_REF):
        subprocess.run(
            ["git", "fetch", "--depth", "1", "origin", TRANSNET_REPO_REF],
            cwd=TRANSNET_REPO,
            check=True,
            capture_output=True,
            env=transnet_env,
        )
        subprocess.run(
            ["git", "checkout", "--detach", TRANSNET_REPO_REF],
            cwd=TRANSNET_REPO,
            check=True,
            capture_output=True,
            env=transnet_env,
        )
TRANSNET_IMPL_DIR = TRANSNET_REPO / "inference"
if not (TRANSNET_IMPL_DIR / "transnetv2.py").is_file():
    raise RuntimeError("Official TransNetV2 inference/transnetv2.py không tồn tại")
sys.path.insert(0, str(TRANSNET_IMPL_DIR))
transnet_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=TRANSNET_REPO, text=True
).strip()
if re.fullmatch(r"[0-9a-f]{40}", TRANSNET_REPO_REF) and transnet_commit != TRANSNET_REPO_REF:
    raise RuntimeError(
        f"TransNet commit mismatch: requested {TRANSNET_REPO_REF}, got {transnet_commit}"
    )

import yaml

freeze = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True
)
dependency_fingerprint = {
    "requirements_sha256": hashlib.sha256(requirements.read_bytes()).hexdigest(),
    "pip_freeze_sha256": hashlib.sha256(freeze.encode("utf-8")).hexdigest(),
    "python": sys.version,
    "transnet_repo_commit": transnet_commit,
}
Path("/kaggle/working/augmentation-dependency-fingerprint.json").write_text(
    json.dumps(dependency_fingerprint, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(f"repo: {git_commit}")
print(f"requirements: {requirements.name}")
print(f"official TransNetV2: {transnet_commit}")
print(f"dependency fingerprint: {dependency_fingerprint['pip_freeze_sha256']}")

## 2 — Dò input read-only và tạo canonical index

SMOKE cần đủ năm nhóm input sau:

1. MP4 của `SMOKE_VIDEO_ID` (bắt buộc attach; Volume BTC không lưu video nguồn);
2. map CSV + thư mục BTC keyframe — attach sẵn, hoặc bật
   `READ_SMOKE_BTC_FROM_MODAL=True` để **chỉ đọc** đúng một video;
3. base flat bundle chứa `data/vectors/siglip_vectors.npz` và các arm dùng khi query;
4. `app.db` của cùng parent bundle;
5. Kaggle Model `hoanggvo/transnetv2-weights` chứa TensorFlow SavedModel
   (`saved_model.pb` và các file `variables.*`). Code inference được clone từ
   official `soCzech/TransNetV2` và commit thực được ghi vào fingerprint.

Dataset Kaggle là read-only. Cell chỉ tạo symlink/copy cache dưới
`/kaggle/working`; không sửa dataset attach và không upload Modal.

In [ ]:
from collections import defaultdict

from mfe.offline.bundle import BundleLayout
from mfe.offline.transport import ModalVolumeTransport

INPUT = Path(INPUT_ROOT)
if not INPUT.is_dir():
    raise FileNotFoundError(f"INPUT_ROOT không tồn tại: {INPUT}")
MODAL_READ_CACHE = Path("/kaggle/working/augmentation_modal_read_cache")
PARENT_MANIFEST_LOCAL = MODAL_READ_CACHE / "parent-manifest.json"
MODAL_MAP_OVERRIDE = None
MODAL_KEYFRAMES_OVERRIDE = None

# Đây là client chỉ-đọc: create_if_missing=False và cell này chỉ gọi get/listdir.
# OUTPUT_TRANSPORT hoàn toàn độc lập, vì vậy SMOKE không thể publish vào Modal.
modal_read = None
if PARENT_MANIFEST_SOURCE == "modal" or READ_SMOKE_BTC_FROM_MODAL:
    modal_read = ModalVolumeTransport(
        VOLUME,
        create_if_missing=False,
        version=VOLUME_VERSION,
    )
if PARENT_MANIFEST_SOURCE == "modal":
    layout = BundleLayout(PARENT_BUNDLE)
    modal_read.get(layout.manifest, PARENT_MANIFEST_LOCAL)
else:
    manifest_source = Path(PARENT_MANIFEST_SOURCE)
    if not manifest_source.is_file():
        raise FileNotFoundError(
            "PARENT_MANIFEST_SOURCE phải là 'modal' hoặc đường dẫn manifest attach sẵn"
        )
    PARENT_MANIFEST_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(manifest_source, PARENT_MANIFEST_LOCAL)

if READ_SMOKE_BTC_FROM_MODAL:
    if SMOKE_VIDEO_ID == "AUTO":
        raise RuntimeError("Fetch BTC từ Modal cần SMOKE_VIDEO_ID cụ thể, không dùng AUTO")
    layout = BundleLayout(PARENT_BUNDLE)
    MODAL_MAP_OVERRIDE = MODAL_READ_CACHE / "map-keyframes" / f"{SMOKE_VIDEO_ID}.csv"
    modal_read.get(layout.map_keyframes(SMOKE_VIDEO_ID), MODAL_MAP_OVERRIDE)
    MODAL_KEYFRAMES_OVERRIDE = MODAL_READ_CACHE / "keyframes" / SMOKE_VIDEO_ID
    remote_files = [
        path for path in modal_read.listdir(layout.keyframes_dir(SMOKE_VIDEO_ID))
        if Path(path).suffix.lower() == ".jpg"
    ]
    if not remote_files:
        raise RuntimeError(f"Modal không có BTC keyframe cho {SMOKE_VIDEO_ID}")
    for remote in remote_files:
        modal_read.get(remote, MODAL_KEYFRAMES_OVERRIDE / Path(remote).name)

INPUT_INDEX = Path("/kaggle/working/augmentation_inputs")
VIDEOS_ROOT = INPUT_INDEX / "videos"
MAPS_ROOT = INPUT_INDEX / "maps"
BTC_KEYFRAMES_ROOT = INPUT_INDEX / "btc_keyframes"
VIDEOS_ROOT.mkdir(parents=True, exist_ok=True)
MAPS_ROOT.mkdir(parents=True, exist_ok=True)
BTC_KEYFRAMES_ROOT.mkdir(parents=True, exist_ok=True)


def _unique_files(pattern: str, *, suffix: str | None = None) -> dict[str, Path]:
    found: dict[str, list[Path]] = defaultdict(list)
    for path in INPUT.rglob(pattern):
        if path.is_file() and (suffix is None or path.suffix.lower() == suffix):
            found[path.stem].append(path)
    duplicates = {key: values for key, values in found.items() if len(values) != 1}
    if duplicates:
        examples = "; ".join(
            f"{key}: {', '.join(map(str, values[:3]))}" for key, values in duplicates.items()
        )
        raise RuntimeError(f"input không duy nhất ({len(duplicates)} tên): {examples}")
    return {key: values[0] for key, values in found.items()}


def _link(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_symlink():
        if destination.resolve() == source.resolve():
            return
        destination.unlink()
    elif destination.exists():
        if os.path.samefile(source, destination):
            return
        raise RuntimeError(f"canonical destination đã tồn tại: {destination}")
    try:
        os.symlink(source, destination, target_is_directory=source.is_dir())
    except OSError as error:
        raise RuntimeError(
            f"không tạo được symlink {destination}; không copy video hàng chục GB"
        ) from error


videos = _unique_files("*.mp4", suffix=".mp4")
if not videos:
    raise RuntimeError("Không thấy .mp4 trong Kaggle Dataset; attach dataset video canonical.")
discovered_video_count = len(videos)
if RUN_MODE == "SMOKE" and SMOKE_VIDEO_ID != "AUTO":
    if SMOKE_VIDEO_ID not in videos:
        raise RuntimeError(f"Không thấy MP4 của smoke video {SMOKE_VIDEO_ID}")
    # Không bắt các video ngoài smoke phải có map/keyframe và không để pipeline
    # nhìn thấy chúng, dù Dataset attach chứa cả batch.
    videos = {SMOKE_VIDEO_ID: videos[SMOKE_VIDEO_ID]}

maps: dict[str, Path] = {}
all_maps: dict[str, list[Path]] = defaultdict(list)
for path in INPUT.rglob("*.csv"):
    if path.is_file():
        all_maps[path.stem].append(path)
for video_id in videos:
    candidates = all_maps.get(video_id, [])
    if video_id == SMOKE_VIDEO_ID and MODAL_MAP_OVERRIDE is not None:
        candidates = [MODAL_MAP_OVERRIDE]
    if len(candidates) != 1:
        raise RuntimeError(f"{video_id}: cần đúng một map-keyframes CSV, thấy {candidates[:3]}")
    maps[video_id] = candidates[0]

keyframes: dict[str, Path] = {}
all_keyframe_dirs: dict[str, list[Path]] = defaultdict(list)
video_names = set(videos)
for path in INPUT.rglob("*"):
    if path.is_dir() and path.name in video_names:
        if any(item.is_file() and item.suffix.lower() == ".jpg" for item in path.iterdir()):
            all_keyframe_dirs[path.name].append(path)
for video_id in videos:
    candidates = all_keyframe_dirs.get(video_id, [])
    if video_id == SMOKE_VIDEO_ID and MODAL_KEYFRAMES_OVERRIDE is not None:
        candidates = [MODAL_KEYFRAMES_OVERRIDE]
    if len(candidates) != 1:
        raise RuntimeError(f"{video_id}: cần đúng một thư mục BTC keyframe, thấy {candidates[:3]}")
    keyframes[video_id] = candidates[0]

for video_id, path in videos.items():
    _link(path, VIDEOS_ROOT / f"{video_id}.mp4")
    _link(maps[video_id], MAPS_ROOT / f"{video_id}.csv")
    _link(keyframes[video_id], BTC_KEYFRAMES_ROOT / video_id)


def _configured_file(glob: str, patterns: tuple[str, ...]) -> Path:
    if glob != "AUTO":
        path = Path(glob)
        if not path.exists():
            raise FileNotFoundError(path)
        if path.is_dir():
            candidates = [
                candidate
                for pattern in patterns
                for candidate in path.rglob(pattern)
                if candidate.is_file()
            ]
            if len(candidates) != 1:
                raise RuntimeError(
                    f"input explicit directory cần đúng một artifact {patterns}, "
                    f"thấy {len(candidates)}"
                )
            return candidates[0]
        return path
    candidates = [path for pattern in patterns for path in INPUT.rglob(pattern) if path.is_file()]
    if len(candidates) != 1:
        raise RuntimeError(f"input cần đúng một artifact {patterns}, thấy {len(candidates)}")
    return candidates[0]


saved_model_pb = _configured_file(SAVED_MODEL_GLOB, ("saved_model.pb",))
saved_model_source = saved_model_pb.parent
source_variables = saved_model_source / "variables"
if source_variables.is_dir():
    variable_files = sorted(path for path in source_variables.iterdir() if path.is_file())
else:
    # Kaggle Model hoanggvo/transnetv2-weights exposes variables.* next to the
    # protobuf. Stage them into the directory layout expected by tf.saved_model.
    variable_files = sorted(saved_model_source.glob("variables.*"))
if len([path for path in variable_files if path.name == "variables.index"]) != 1:
    raise RuntimeError(f"SavedModel thiếu/nhân đôi variables.index: {variable_files}")
if not any(path.name.startswith("variables.data-") for path in variable_files):
    raise RuntimeError(f"SavedModel thiếu variables.data shard: {variable_files}")

checkpoint_target = Path("/kaggle/working/transnetv2_saved_model")
_link(saved_model_pb, checkpoint_target / "saved_model.pb")
for source in variable_files:
    _link(source, checkpoint_target / "variables" / source.name)
assets = saved_model_source / "assets"
if assets.is_dir():
    for source in sorted(path for path in assets.rglob("*") if path.is_file()):
        _link(source, checkpoint_target / "assets" / source.relative_to(assets))

flat_marker = _configured_file(BASE_FLAT_GLOB, ("siglip_vectors.npz",))
BASE_FLAT_ROOT = (
    flat_marker.parent.parent.parent
    if flat_marker.parent.name == "vectors"
    else flat_marker.parent
)
base_db_source = _configured_file(BASE_DB_GLOB, ("app.db",))
BASE_DB = base_db_source

print(f"videos discovered/active: {discovered_video_count}/{len(videos)}")
print(f"maps: {len(maps)}")
print(f"BTC keyframes: {BTC_KEYFRAMES_ROOT}")
print(f"TensorFlow SavedModel: {checkpoint_target}")
print(f"TransNet implementation: {TRANSNET_IMPL_DIR} @ {transnet_commit}")
print(f"base flat: {BASE_FLAT_ROOT}")
print(f"base SQLite: {BASE_DB}")

## 3 — Preflight, config hash và transport

In [ ]:
from mfe.offline.augmentation.config import load_augmentation_config
from mfe.offline.augmentation.acceptance import (
    build_blocked_oracle_evidence,
    validate_primary_equal_budget,
    validate_worker_assignment,
    write_json_atomic as write_acceptance_json,
)
from mfe.offline.augmentation.io import (
    read_jsonl,
    sha256_path,
    write_json_atomic,
    write_jsonl_atomic,
)
from mfe.offline.augmentation.locking import load_inventory
from mfe.offline.augmentation.store import ExperimentStore
from mfe.offline.transport import LocalDirTransport, build_transport

CONFIG_PATH = REPO / "config/keyframe_augmentation_runtime_v1.yaml"
validate_worker_assignment(MEMBER, WORKER)
raw_config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
if raw_config["experiment"]["parent_bundle"] != PARENT_BUNDLE:
    raise RuntimeError("PARENT_BUNDLE không khớp config đã review")

# Chọn video trước khi tạo Gate 0. SMOKE chỉ mang đúng query của video này;
# held-out runtime là file rỗng nên không có đường vô tình mở held-out.
full_dev_queryset = REPO / raw_config["dataset"]["dev_queryset"]
full_heldout_queryset = REPO / raw_config["dataset"]["heldout_queryset"]
full_dev_rows = read_jsonl(full_dev_queryset)
full_heldout_rows = read_jsonl(full_heldout_queryset)
full_dev_ids = {str(row["gt_video_id"]) for row in full_dev_rows if row.get("gt_video_id")}
full_heldout_ids = {
    str(row["gt_video_id"]) for row in full_heldout_rows if row.get("gt_video_id")
}
if full_dev_ids & full_heldout_ids:
    raise RuntimeError(f"DEV/held-out overlap: {sorted(full_dev_ids & full_heldout_ids)}")


def max_literal_gap(video_id: str) -> float:
    """Read literal pts_time values from CSV; never infer them from ordinal/FPS."""
    import csv

    with (MAPS_ROOT / f"{video_id}.csv").open(
        "r", encoding="utf-8-sig", newline=""
    ) as stream:
        rows = list(csv.DictReader(stream))
    pts = sorted(float(row["pts_time"]) for row in rows)
    return max((right - left for left, right in zip(pts, pts[1:])), default=0.0)


available_dev_ids = sorted(full_dev_ids & set(videos))
if RUN_MODE == "SMOKE":
    if not available_dev_ids:
        raise RuntimeError("Không có video DEV nào trong input để smoke test")
    if SMOKE_VIDEO_ID == "AUTO":
        SMOKE_VIDEO_ID = sorted(
            available_dev_ids, key=lambda item: (-max_literal_gap(item), item)
        )[0]
    if SMOKE_VIDEO_ID not in full_dev_ids:
        raise RuntimeError(f"SMOKE_VIDEO_ID phải thuộc DEV: {SMOKE_VIDEO_ID}")
    if SMOKE_VIDEO_ID in full_heldout_ids:
        raise RuntimeError(f"SMOKE_VIDEO_ID không được thuộc held-out: {SMOKE_VIDEO_ID}")
    if SMOKE_VIDEO_ID not in videos:
        raise RuntimeError(f"Input thiếu video smoke {SMOKE_VIDEO_ID}.mp4")

    active_index = Path("/kaggle/working/augmentation_smoke_input") / SMOKE_VIDEO_ID
    ACTIVE_VIDEOS_ROOT = active_index / "videos"
    ACTIVE_MAPS_ROOT = active_index / "maps"
    ACTIVE_BTC_KEYFRAMES_ROOT = active_index / "btc_keyframes"
    _link(VIDEOS_ROOT / f"{SMOKE_VIDEO_ID}.mp4", ACTIVE_VIDEOS_ROOT / f"{SMOKE_VIDEO_ID}.mp4")
    _link(MAPS_ROOT / f"{SMOKE_VIDEO_ID}.csv", ACTIVE_MAPS_ROOT / f"{SMOKE_VIDEO_ID}.csv")
    _link(BTC_KEYFRAMES_ROOT / SMOKE_VIDEO_ID, ACTIVE_BTC_KEYFRAMES_ROOT / SMOKE_VIDEO_ID)

    runtime_querysets = Path("/kaggle/working/augmentation_runtime_querysets") / SMOKE_VIDEO_ID
    runtime_dev = runtime_querysets / "dev.jsonl"
    runtime_heldout = runtime_querysets / "heldout.jsonl"
    smoke_rows = [row for row in full_dev_rows if row.get("gt_video_id") == SMOKE_VIDEO_ID]
    if not smoke_rows:
        raise RuntimeError(f"DEV không có query cho {SMOKE_VIDEO_ID}")
    write_jsonl_atomic(runtime_dev, smoke_rows)
    write_jsonl_atomic(runtime_heldout, [])
    raw_config["dataset"]["dev_queryset"] = str(runtime_dev)
    raw_config["dataset"]["heldout_queryset"] = str(runtime_heldout)
    raw_config["dataset"]["extra_pilot_videos"] = 0
else:
    ACTIVE_VIDEOS_ROOT = VIDEOS_ROOT
    ACTIVE_MAPS_ROOT = MAPS_ROOT
    ACTIVE_BTC_KEYFRAMES_ROOT = BTC_KEYFRAMES_ROOT
    missing = (full_dev_ids | full_heldout_ids) - set(videos)
    if missing:
        raise RuntimeError(f"PILOT input thiếu {len(missing)} video queryset")

# The checked-in config uses stable repository-relative paths, while Kaggle
# stores the immutable base bundle in a read-only Dataset.  Stage symlinks at a
# fixed working path and put those paths in the runtime config so every CLI
# command (including evaluate-dev/oracle) reads the same discovered base.
BASE_FLAT_LINK = INPUT_INDEX / "base_flat"
BASE_DB_LINK = INPUT_INDEX / "base_relational.db"
_link(BASE_FLAT_ROOT, BASE_FLAT_LINK)
_link(BASE_DB, BASE_DB_LINK)
raw_config["paths"]["base_flat_root"] = str(BASE_FLAT_LINK)
raw_config["paths"]["base_relational_db"] = str(BASE_DB_LINK)
expected_checkpoint_sha = str(raw_config["detector"]["checkpoint_sha256"])
actual_checkpoint_sha = sha256_path(checkpoint_target)
if RUN_MODE == "SMOKE" and SMOKE_CHECKPOINT_SHA256 == "AUTO":
    raw_config["detector"]["checkpoint_sha256"] = actual_checkpoint_sha
    expected_checkpoint_sha = actual_checkpoint_sha
elif RUN_MODE == "SMOKE" and SMOKE_CHECKPOINT_SHA256:
    raw_config["detector"]["checkpoint_sha256"] = SMOKE_CHECKPOINT_SHA256
    expected_checkpoint_sha = SMOKE_CHECKPOINT_SHA256
if expected_checkpoint_sha == "0" * 64:
    raise RuntimeError(
        "PILOT yêu cầu Lead pin SHA256 checkpoint thật trong config."
    )
if actual_checkpoint_sha != expected_checkpoint_sha:
    raise RuntimeError(
        f"TransNet checkpoint SHA mismatch: expected {expected_checkpoint_sha}, "
        f"got {actual_checkpoint_sha}"
    )

# Smoke dùng experiment root riêng để không làm bẩn pilot đã lock.
if RUN_MODE == "SMOKE":
    smoke_id = f"{EXPERIMENT_ID}_smoke_{git_commit[:7]}"
    raw_config["experiment"]["id"] = smoke_id
    raw_config["paths"]["workdir"] = f"/kaggle/working/augmentation_work/{smoke_id}"
    raw_config["transport"]["local_root"] = f"/kaggle/working/augmentation_local/{smoke_id}"
else:
    smoke_id = EXPERIMENT_ID
raw_config["transport"]["kind"] = OUTPUT_TRANSPORT
raw_config["transport"]["volume"] = VOLUME
raw_config["transport"]["version"] = VOLUME_VERSION
raw_config["detector"]["checkpoint"] = str(checkpoint_target)
raw_config["paths"]["eval_config"] = str(REPO / raw_config["paths"]["eval_config"])
raw_config["dataset"]["video_inventory"] = (
    f"/kaggle/working/augmentation_inventory/{smoke_id}.jsonl"
)
CONFIG_PATH = Path("/kaggle/working") / f"runtime_{RUN_MODE.lower()}_augmentation.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(raw_config, sort_keys=False), encoding="utf-8")

config = load_augmentation_config(CONFIG_PATH)
WORK = config.resolve(config.paths.workdir)
WORK.mkdir(parents=True, exist_ok=True)
transport = build_transport(
    config.transport.kind,
    volume=config.transport.volume,
    local_root=str(config.resolve(config.transport.local_root)),
    version=config.transport.version,
)
if RUN_MODE == "SMOKE" and not isinstance(transport, LocalDirTransport):
    raise RuntimeError("SAFETY: SMOKE output transport không phải LocalDirTransport")
store = ExperimentStore(transport, config.experiment.id, WORK)

# Lệnh lock đọc parent manifest qua output transport. Với local smoke, seed một
# bản sao vào sandbox local; đây không phải upload và không chạm Volume Modal.
if RUN_MODE == "SMOKE":
    local_parent_manifest = (
        config.resolve(config.transport.local_root)
        / "bundle" / config.experiment.parent_bundle / "manifest.json"
    )
    local_parent_manifest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(PARENT_MANIFEST_LOCAL, local_parent_manifest)

dependency_path = Path("/kaggle/working/augmentation-dependency-fingerprint.json")
if dependency_path.is_file():
    store.publish(dependency_path, "gate0/dependency-fingerprint.json")

dev_queryset = config.resolve(config.dataset.dev_queryset)
heldout_queryset = config.resolve(config.dataset.heldout_queryset)
dev_ids = {str(row["gt_video_id"]) for row in read_jsonl(dev_queryset) if row.get("gt_video_id")}
heldout_ids = {
    str(row["gt_video_id"])
    for row in read_jsonl(heldout_queryset)
    if row.get("gt_video_id")
}
if RUN_MODE == "SMOKE" and (dev_ids != {SMOKE_VIDEO_ID} or heldout_ids):
    raise RuntimeError("SAFETY: smoke runtime phải có đúng một DEV video và held-out rỗng")

print(f"config: {CONFIG_PATH}")
print(f"config hash: {config.canonical_hash()}")
print(f"owner/worker: {MEMBER}/{WORKER}")
print(f"output transport: {type(transport).__name__}")
print(f"experiment artifacts: {config.resolve(config.transport.local_root) if RUN_MODE == 'SMOKE' else store.paths.root}")
print(f"smoke video/query: {SMOKE_VIDEO_ID} / {len(read_jsonl(dev_queryset))}")

## 4 — Restore, inventory và Gate 0

In [ ]:
def run_cli(*arguments: str) -> None:
    """Run one tested augmentation CLI command with the cloned source tree."""
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO / "src") + os.pathsep + env.get("PYTHONPATH", "")
    if TRANSNET_IMPL_DIR is not None:
        env["PYTHONPATH"] = str(TRANSNET_IMPL_DIR) + os.pathsep + env["PYTHONPATH"]
    command = [sys.executable, "-m", "mfe.offline.augmentation.cli", *arguments]
    print("$", " ".join(command))
    subprocess.run(command, cwd=REPO, env=env, check=True)


def run_cli_once(arguments: tuple[str, ...], outputs: tuple[Path, ...]) -> None:
    """Skip a completed atomic stage, otherwise execute it normally."""
    if outputs and all(path.exists() for path in outputs):
        print("resume: outputs already present —", arguments[0])
        return
    run_cli(*arguments)


def checkpoint(label: str, status: str = "complete", **extra) -> None:
    payload = {
        "experiment_id": config.experiment.id,
        "config_hash": config.canonical_hash(),
        "git_commit": git_commit,
        "label": label,
        "status": status,
        "updated_at_unix": time.time(),
        **extra,
    }
    path = write_json_atomic(WORK / "run-state.json", payload)
    store.publish(path, "run-state.json")


def restore_delta_aliases() -> None:
    """Expose published delta flats at the paths consumed by the CLI/evaluator."""
    for cohort in ("dev", "heldout"):
        remote_root = WORK / "gate2" / cohort / "delta-flat"
        if not remote_root.is_dir():
            continue
        for source in remote_root.iterdir():
            if not source.is_dir():
                continue
            target = WORK / "delta-merge" / cohort / source.name
            if target.exists() or target.is_symlink():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            try:
                os.symlink(source, target, target_is_directory=True)
            except OSError as error:
                raise RuntimeError(
                    f"không tạo được alias delta {target}; không copy flat bundle lớn"
                ) from error


# SMOKE restores only from its local sandbox. PILOT may restore from Modal only
# after the explicit write gates in cell 0 have been opened by Lead.
restored = store.restore_prefix("", WORK, verify=True)
restore_delta_aliases()
inventory_path = config.resolve(config.dataset.video_inventory)
remote_inventory = WORK / "gate0" / "video_inventory.jsonl"
if not inventory_path.is_file() and remote_inventory.is_file():
    inventory_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(remote_inventory, inventory_path)
if not inventory_path.is_file():
    run_cli(
        "inventory",
        "--videos-root", str(ACTIVE_VIDEOS_ROOT),
        "--maps-root", str(ACTIVE_MAPS_ROOT),
        "--output", str(inventory_path),
    )
    store.publish(inventory_path, "gate0/video_inventory.jsonl")

lock_path = WORK / "gate0" / "lock.json"
if not lock_path.is_file():
    run_cli("lock", "--config", str(CONFIG_PATH))
else:
    print(f"resume Gate 0: {lock_path}")

inventory = load_inventory(inventory_path, verify_files=True)
if RUN_MODE == "PILOT":
    cohort = json.loads(lock_path.read_text(encoding="utf-8"))["cohort"]
    if len(cohort) != len(dev_ids) + config.dataset.extra_pilot_videos:
        raise RuntimeError(f"Gate 0 cohort bất thường: {len(cohort)} video")
checkpoint("gate0", cohort_size=len(json.loads(lock_path.read_text(encoding="utf-8"))["cohort"]))
print(f"restored: {len(restored)} artifact")
print(f"SMOKE_VIDEO_ID: {SMOKE_VIDEO_ID}")
run_cli("status", "--config", str(CONFIG_PATH))

## 5 — SMOKE: một video end-to-end

In [ ]:
def grid_name(threshold: float, gap: float, k_max: int) -> str:
    token = lambda value: str(value).replace(".", "p")
    return f"t{token(threshold)}_g{token(gap)}_k{k_max}"


def smoke_pipeline() -> None:
    if RUN_MODE != "SMOKE":
        print("PILOT mode: bỏ qua cell smoke riêng.")
        return
    grid = grid_name(DEFAULT_THRESHOLD, DEFAULT_TARGET_GAP, DEFAULT_K_MAX)
    audit = WORK / "gate1/dev/audit" / f"{grid}.jsonl"
    selectors = WORK / "gate2/dev/selectors" / grid
    run_cli(
        "detect", "--config", str(CONFIG_PATH), "--cohort", "dev",
        "--video", SMOKE_VIDEO_ID, "--video-root", str(ACTIVE_VIDEOS_ROOT),
    )
    run_cli(
        "audit", "--config", str(CONFIG_PATH), "--cohort", "dev",
        "--video", SMOKE_VIDEO_ID, "--threshold", str(DEFAULT_THRESHOLD),
        "--target-gap", str(DEFAULT_TARGET_GAP), "--k-max", str(DEFAULT_K_MAX),
        "--maps-root", str(ACTIVE_MAPS_ROOT),
    )
    run_cli(
        "select", "--config", str(CONFIG_PATH), "--cohort", "dev",
        "--video", SMOKE_VIDEO_ID, "--threshold", str(DEFAULT_THRESHOLD),
        "--target-gap", str(DEFAULT_TARGET_GAP), "--k-max", str(DEFAULT_K_MAX),
        "--maps-root", str(ACTIVE_MAPS_ROOT),
    )
    b3 = selectors / "b3.jsonl"
    b4_pool = selectors / "b4_pool.jsonl"
    if not b3.is_file() or not read_jsonl(b3):
        report = write_json_atomic(WORK / "smoke-report.json", {
            "status": "NO_COVERAGE_GAP",
            "effectiveness_signal": "INCONCLUSIVE",
            "video_id": SMOKE_VIDEO_ID,
            "config_hash": config.canonical_hash(),
            "checkpoint_sha256": actual_checkpoint_sha,
            "output_transport": type(transport).__name__,
            "message": "Video không có candidate C0/C1 dưới grid smoke; không sinh frame giả.",
        })
        store.publish(report, "smoke-report.json")
        checkpoint("smoke", status="NO_COVERAGE_GAP", report=str(report))
        print(report.read_text(encoding="utf-8"))
        return

    for variant, candidates in (("b3", b3), ("b4_pool", b4_pool)):
        materialized = WORK / f"pool/dev/{variant}/materialized.jsonl"
        manifest = WORK / f"pool/dev/{variant}/augmentation_manifest.jsonl"
        run_cli_once(
            (
                "materialize", "--config", str(CONFIG_PATH), "--cohort", "dev",
                "--variant", variant, "--candidates", str(candidates),
                "--video-root", str(ACTIVE_VIDEOS_ROOT), "--maps-root", str(ACTIVE_MAPS_ROOT),
            ),
            (materialized, manifest),
        )
        delta_root = WORK / f"delta-merge/dev/{variant}/out"
        run_cli_once(
            (
                "extract-features", "--config", str(CONFIG_PATH), "--cohort", "dev",
                "--variant", variant, "--candidates", str(candidates),
                "--stages", FEATURE_STAGES, "--video-root", str(ACTIVE_VIDEOS_ROOT),
            ),
            (delta_root / "manifest.json", delta_root.parent / "merge-report.json"),
        )

    dedup_paths = {}
    for variant, candidates in (("b3", b3), ("b4_pool", b4_pool)):
        materialized = WORK / f"pool/dev/{variant}/materialized.jsonl"
        inputs = WORK / f"gate2/dev/dedup-inputs/{variant}"
        run_cli(
            "prepare-dedup", "--config", str(CONFIG_PATH), "--cohort", "dev",
            "--variant", variant, "--candidates", str(candidates), "--audit", str(audit),
            "--materialized", str(materialized), "--btc-keyframes-root", str(ACTIVE_BTC_KEYFRAMES_ROOT),
            "--base-flat-root", str(BASE_FLAT_ROOT),
            "--delta-flat-root", str(WORK / f"delta-merge/dev/{variant}/out"),
            "--maps-root", str(ACTIVE_MAPS_ROOT), "--output-dir", str(inputs),
        )
        kept = WORK / f"gate2/dev/dedup/{variant}-kept.jsonl"
        run_cli(
            "dedup", "--config", str(CONFIG_PATH), "--cohort", "dev",
            "--candidates", str(candidates), "--btc-shots", str(inputs / "btc-shots.jsonl"),
            "--btc-phashes", str(inputs / "btc-phashes.json"),
            "--btc-embeddings", str(inputs / "btc-embeddings.npz"),
            "--candidate-phashes", str(inputs / "candidate-phashes.json"),
            "--candidate-embeddings", str(inputs / "candidate-embeddings.npz"),
            "--output", str(kept),
        )
        dedup_paths[variant] = kept

    b4 = WORK / "gate2/dev/dedup/b4-matched.jsonl"
    run_cli(
        "match-control", "--config", str(CONFIG_PATH), "--cohort", "dev",
        "--b3", str(dedup_paths["b3"]), "--b4-pool", str(dedup_paths["b4_pool"]),
        "--output", str(b4),
    )
    b3_db = WORK / "gate2/dev/snapshots/smoke-b3.db"
    b4_db = WORK / "gate2/dev/snapshots/smoke-b4.db"
    for candidates, output in ((dedup_paths["b3"], b3_db), (b4, b4_db)):
        run_cli(
            "snapshot", "--config", str(CONFIG_PATH), "--cohort", "dev",
            "--base-db", str(BASE_DB), "--candidates", str(candidates),
            "--output", str(output),
        )

    # Score only the selected video's queries as a smoke signal. This is not a
    # DEV Go/No-Go decision; the full pilot cell performs official A/B.
    smoke_queries = WORK / "smoke-queryset.jsonl"
    queries = [row for row in read_jsonl(dev_queryset) if row.get("gt_video_id") == SMOKE_VIDEO_ID]
    write_jsonl_atomic(smoke_queries, queries)
    from mfe.offline.augmentation.evaluation import run_overlay_variant

    delta_b3 = WORK / "delta-merge/dev/b3/out"
    delta_b4 = WORK / "delta-merge/dev/b4_pool/out"
    runs = {}
    for name, relational, delta, include in (
        ("B0", BASE_DB, delta_b3, None),
        ("B3", b3_db, delta_b3, dedup_paths["b3"]),
        ("B4", b4_db, delta_b4, b4),
    ):
        runs[name] = run_overlay_variant(
            name=name,
            config_path=str(REPO / "config/pilot.yaml"),
            queryset_path=str(smoke_queries),
            relational_db=str(relational),
            base_flat_root=str(BASE_FLAT_ROOT),
            delta_flat_root=str(delta),
            include_ids=[] if include is None else [row["keyframe_id"] for row in read_jsonl(include)],
            latency_repeats=LATENCY_REPEATS,
        )
    b3_delta_b0 = runs["B3"].mean_score - runs["B0"].mean_score
    b3_delta_b4 = runs["B3"].mean_score - runs["B4"].mean_score
    latency_ratio = (
        runs["B3"].p95_latency_ms / runs["B0"].p95_latency_ms
        if runs["B0"].p95_latency_ms > 0 else None
    )
    effectiveness_signal = (
        "PROMISING" if b3_delta_b0 > 0 and b3_delta_b4 > 0
        else "NO_GAIN" if b3_delta_b0 <= 0 and b3_delta_b4 <= 0
        else "MIXED"
    )
    report = write_json_atomic(WORK / "smoke-report.json", {
        "status": "PASS",
        "effectiveness_signal": effectiveness_signal,
        "warning": "One-video smoke signal only; this is not the DEV Go/No-Go gate.",
        "video_id": SMOKE_VIDEO_ID,
        "config_hash": config.canonical_hash(),
        "checkpoint_sha256": actual_checkpoint_sha,
        "output_transport": type(transport).__name__,
        "query_count": len(queries),
        "query_types": sorted({str(row.get("type", "unknown")) for row in queries}),
        "candidate_counts": {
            "raw_b3": len(read_jsonl(b3)),
            "raw_b4_pool": len(read_jsonl(b4_pool)),
            "kept_b3": len(read_jsonl(dedup_paths["b3"])),
            "kept_b4_pool": len(read_jsonl(dedup_paths["b4_pool"])),
            "matched_b4": len(read_jsonl(b4)),
        },
        "effect_deltas": {
            "b3_minus_b0": b3_delta_b0,
            "b3_minus_matched_b4": b3_delta_b4,
            "b3_latency_ratio": latency_ratio,
        },
        "variants": {
            name: {"mean_score": run.mean_score, "p95_latency_ms": run.p95_latency_ms}
            for name, run in runs.items()
        },
        "base_flat_root": str(BASE_FLAT_ROOT),
        "base_sqlite": str(BASE_DB),
    })
    store.publish(report, "smoke-report.json")
    checkpoint("smoke", report=str(report))
    print(report.read_text(encoding="utf-8"))


smoke_pipeline()

## 6 — PILOT Gate 1: detector, manual calibration, audit và oracle

In [ ]:
def pilot_gate1() -> None:
    if RUN_MODE != "PILOT":
        print("SMOKE mode: bỏ qua Gate 1 pilot.")
        return
    run_cli(
        "detect", "--config", str(CONFIG_PATH), "--cohort", "dev",
        "--video-root", str(VIDEOS_ROOT),
    )
    run_cli(
        "audit", "--config", str(CONFIG_PATH), "--cohort", "dev",
        "--maps-root", str(MAPS_ROOT),
    )
    labels = REPO / LABELS_PATH
    labels.parent.mkdir(parents=True, exist_ok=True)
    if not labels.is_file():
        template = WORK / "gate1/dev/manual-labels-template.jsonl"
        template.write_text(
            "# Sửa null thành true/false sau khi xem boundary; cần cả positive và negative.\n"
            "# Mỗi dòng: {\"video_id\":\"L21_V001\",\"frame_idx\":1234,\"is_boundary\":true}\n",
            encoding="utf-8",
        )
        store.publish(template, "gate1/dev/manual-labels-template.jsonl")
        checkpoint("gate1", status="WAITING_FOR_LABELS", template=str(template))
        print(f"Cần manual labels: {template}. Sau khi hoàn tất, đặt file tại {labels} rồi chạy lại.")
        return
    run_cli("calibrate", "--config", str(CONFIG_PATH), "--labels", str(labels))
    calibration = json.loads(
        (WORK / "gate1/dev/detector-calibration.json").read_text(encoding="utf-8")
    )
    threshold = float(calibration["selected_threshold"])
    oracle_audit = WORK / "gate1/dev/audit" / f"{grid_name(threshold, DEFAULT_TARGET_GAP, DEFAULT_K_MAX)}.jsonl"
    if not oracle_audit.is_file():
        run_cli(
            "audit", "--config", str(CONFIG_PATH), "--cohort", "dev",
            "--threshold", str(threshold), "--target-gap", str(DEFAULT_TARGET_GAP),
            "--k-max", str(DEFAULT_K_MAX), "--maps-root", str(MAPS_ROOT),
        )
    run_cli(
        "oracle", "--config", str(CONFIG_PATH), "--cohort", "dev",
        "--threshold", str(threshold), "--target-gap", str(DEFAULT_TARGET_GAP),
        "--k-max", str(DEFAULT_K_MAX), "--maps-root", str(MAPS_ROOT),
    )
    oracle_candidates = WORK / "gate1/dev/oracles/candidates.jsonl"
    if not oracle_candidates.is_file() or not read_jsonl(oracle_candidates):
        checkpoint("gate1", status="NO_GO_ORACLE", reason="oracle không có target")
        print("NO_GO_ORACLE: không có oracle target hợp lệ; giữ BTC-only.")
        return

    # Runtime oracle của Trọng chọn frame xa BTC nhất theo thời gian. Nó hữu ích
    # như O1-T diagnostic nhưng KHÔNG phải O1-S. O1-S phải decode toàn bộ GT
    # interval, chạy feature parity + frozen retrieval thật và chọn best source.
    # Vì notebook chưa có artifact đó, ghi BLOCKED có schema thay vì suy điểm.
    canonical_events = []
    for query in read_jsonl(dev_queryset):
        query_type = str(query.get("type") or query.get("query_type") or "KIS").upper()
        if query_type in {"Q&A", "QNA"}:
            query_type = "QA"
        intervals = (
            query.get("gt_events_strict")
            or query.get("gt_events")
            or [query.get("gt_frame_range_strict") or query.get("gt_frame_range")]
        )
        for interval in intervals:
            if not interval:
                continue
            canonical_events.append({
                "query_id": str(query["query_id"]),
                "query_type": query_type,
                "video_id": str(query["gt_video_id"]),
                "start_frame": int(interval[0]),
                "end_frame": int(interval[1]),
            })
    blocked_rows = build_blocked_oracle_evidence(
        canonical_events,
        provenance_hashes={
            "runtime_config_hash": config.canonical_hash(),
            "checkpoint_sha256": actual_checkpoint_sha,
            "git_commit": git_commit,
        },
        failure_stage="representation",
    )
    blocked_path = write_acceptance_json(
        WORK / "gate1/dev/oracles/o1s-blocked.json", blocked_rows
    )
    status_path = write_acceptance_json(
        WORK / "gate1/dev/oracles/canonical-status.json",
        {
            "status": "BLOCKED_MISSING_O1_S_FEATURE_RANK_ARTIFACT",
            "runtime_oracle_authority": "O1-T_DIAGNOSTIC_ONLY",
            "o1s_rows": len(blocked_rows),
            "o1s_artifact": str(blocked_path),
            "heldout_accessed": False,
        },
    )
    store.publish(blocked_path, "gate1/dev/oracles/o1s-blocked.json")
    store.publish(status_path, "gate1/dev/oracles/canonical-status.json")
    checkpoint(
        "gate1",
        status="BLOCKED_MISSING_O1_S_FEATURE_RANK_ARTIFACT",
        canonical_status=str(status_path),
    )
    print(status_path.read_text(encoding="utf-8"))
    return
    oracle_materialized = WORK / "pool/dev/oracle/materialized.jsonl"
    oracle_manifest = WORK / "pool/dev/oracle/augmentation_manifest.jsonl"
    run_cli_once(
        (
            "materialize", "--config", str(CONFIG_PATH), "--cohort", "dev", "--variant", "oracle",
            "--candidates", str(oracle_candidates), "--video-root", str(VIDEOS_ROOT),
            "--maps-root", str(MAPS_ROOT),
        ),
        (oracle_materialized, oracle_manifest),
    )
    oracle_delta = WORK / "delta-merge/dev/oracle/out"
    run_cli_once(
        (
            "extract-features", "--config", str(CONFIG_PATH), "--cohort", "dev", "--variant", "oracle",
            "--candidates", str(oracle_candidates), "--stages", FEATURE_STAGES,
            "--video-root", str(VIDEOS_ROOT),
        ),
        (oracle_delta / "manifest.json", oracle_delta.parent / "merge-report.json"),
    )
    oracle_db = WORK / "gate1/dev/oracles/oracle.db"
    run_cli(
        "snapshot", "--config", str(CONFIG_PATH), "--cohort", "dev", "--base-db", str(BASE_DB),
        "--candidates", str(oracle_candidates), "--output", str(oracle_db),
    )
    run_cli(
        "evaluate-oracle", "--config", str(CONFIG_PATH),
        "--oracle-delta-root", str(WORK / "delta-merge/dev/oracle/out"),
        "--base-relational", str(BASE_DB), "--oracle-relational", str(oracle_db),
        "--oracle-candidates", str(oracle_candidates),
        "--oracle-targets", str(WORK / "gate1/dev/oracles/targets.jsonl"),
        "--repeats", str(LATENCY_REPEATS),
    )
    oracle_report = WORK / "gate1/dev/oracles/report.json"
    gate = json.loads(oracle_report.read_text(encoding="utf-8"))["gate"]
    checkpoint("gate1", status="PASS" if gate.get("go") else "NO_GO_ORACLE", oracle_gate=gate)
    print(json.dumps(gate, ensure_ascii=False, indent=2))


pilot_gate1()

## Manual calibration — bắt buộc trước Gate 2

Sau khi cell Gate 1 chạy xong lần đầu:

1. tải `gate1/dev/manual-labels-template.jsonl` từ Modal (notebook đã publish file này);
2. xem boundary trên video, sửa `null` thành `true`/`false`;
3. lưu đúng `augmentation_labels/transnet_dev.jsonl` trong repo hoặc đặt
   `LABELS_PATH` thành đường dẫn tuyệt đối trong `/kaggle/input/...`;
4. chạy lại notebook Version mới.

Không được gán tất cả `false`, không dùng held-out để label, và không sửa
threshold sau khi đã mở held-out. Nếu oracle gate không đạt, kết quả đúng là
`NO_GO_ORACLE`, không phải lỗi cần ép chạy tiếp.

## 7 — PILOT Gate 2: B1/B2/B3/B4, dedup, snapshot, A/B và freeze

In [ ]:
def pilot_gate2() -> None:
    if RUN_MODE != "PILOT":
        print("SMOKE mode: không chạy pilot Gate 2.")
        return
    calibration_path = WORK / "gate1/dev/detector-calibration.json"
    oracle_status_path = WORK / "gate1/dev/oracles/canonical-status.json"
    if not calibration_path.is_file() or not oracle_status_path.is_file():
        print("Gate 2 đang chờ calibration/O1-S canonical; chạy lại sau cell Gate 1.")
        return
    oracle_status = json.loads(oracle_status_path.read_text(encoding="utf-8"))
    if oracle_status.get("status") != "MEASURED_O1_S_O1_R_PASS":
        checkpoint("gate2", status="BLOCKED_O1_S_O1_R", oracle_status=oracle_status)
        print("BLOCKED_O1_S_O1_R: chưa được mở official B0–B4.")
        return

    threshold = float(json.loads(calibration_path.read_text(encoding="utf-8"))["selected_threshold"])
    grid_values = [
        (threshold, float(gap), int(k_max))
        for gap in config.coverage.target_gap_seconds_grid
        for k_max in config.coverage.k_max_grid
    ]
    selected = {}
    union_by_id = {}
    for current_threshold, gap, k_max in grid_values:
        grid = grid_name(current_threshold, gap, k_max)
        try:
            run_cli(
                "select", "--config", str(CONFIG_PATH), "--cohort", "dev",
                "--threshold", str(current_threshold), "--target-gap", str(gap),
                "--k-max", str(k_max), "--maps-root", str(MAPS_ROOT),
            )
        except subprocess.CalledProcessError as error:
            print(f"skip grid {grid}: selector failed with exit {error.returncode}")
            continue
        selector_dir = WORK / "gate2/dev/selectors" / grid
        selected[grid] = {
            "threshold": current_threshold,
            "gap": gap,
            "k_max": k_max,
            "dir": selector_dir,
            "audit": WORK / "gate1/dev/audit" / f"{grid}.jsonl",
        }
        for name in ("b1", "b2", "b3", "b4_pool"):
            path = selector_dir / f"{name}.jsonl"
            if path.is_file():
                for row in read_jsonl(path):
                    union_by_id.setdefault(row["keyframe_id"], row)
    if not union_by_id:
        checkpoint("gate2", status="NO_GO_NO_CANDIDATES")
        print("NO_GO_NO_CANDIDATES: không có C0/C1 candidate dưới cap.")
        return

    union_candidates = WORK / "pool/dev/union-candidates.jsonl"
    write_jsonl_atomic(union_candidates, [union_by_id[key] for key in sorted(union_by_id)])
    union_materialized = WORK / "pool/dev/union/materialized.jsonl"
    union_manifest = WORK / "pool/dev/union/augmentation_manifest.jsonl"
    run_cli_once(
        (
            "materialize", "--config", str(CONFIG_PATH), "--cohort", "dev", "--variant", "union",
            "--candidates", str(union_candidates), "--video-root", str(VIDEOS_ROOT),
            "--maps-root", str(MAPS_ROOT),
        ),
        (union_materialized, union_manifest),
    )
    union_delta = WORK / "delta-merge/dev/union/out"
    run_cli_once(
        (
            "extract-features", "--config", str(CONFIG_PATH), "--cohort", "dev", "--variant", "union",
            "--candidates", str(union_candidates), "--stages", FEATURE_STAGES,
            "--video-root", str(VIDEOS_ROOT),
        ),
        (union_delta / "manifest.json", union_delta.parent / "merge-report.json"),
    )

    def dedup_variant(grid: str, name: str, candidates: Path) -> tuple[Path, Path]:
        label = f"{grid}-{name}"
        inputs = WORK / "gate2/dev/dedup-inputs" / label
        run_cli(
            "prepare-dedup", "--config", str(CONFIG_PATH), "--cohort", "dev",
            "--variant", label, "--candidates", str(candidates),
            "--audit", str(selected[grid]["audit"]), "--materialized", str(union_materialized),
            "--btc-keyframes-root", str(BTC_KEYFRAMES_ROOT), "--base-flat-root", str(BASE_FLAT_ROOT),
            "--delta-flat-root", str(union_delta), "--maps-root", str(MAPS_ROOT),
            "--output-dir", str(inputs),
        )
        kept = WORK / "gate2/dev/dedup" / f"{label}-kept.jsonl"
        run_cli(
            "dedup", "--config", str(CONFIG_PATH), "--cohort", "dev",
            "--candidates", str(candidates), "--btc-shots", str(inputs / "btc-shots.jsonl"),
            "--btc-phashes", str(inputs / "btc-phashes.json"),
            "--btc-embeddings", str(inputs / "btc-embeddings.npz"),
            "--candidate-phashes", str(inputs / "candidate-phashes.json"),
            "--candidate-embeddings", str(inputs / "candidate-embeddings.npz"),
            "--output", str(kept),
        )
        return kept, WORK / "gate2/dev/dedup" / f"{label}-report.json"

    reports = []
    grid_reports_dir = WORK / "gate2/dev/grid-reports"
    grid_reports_dir.mkdir(parents=True, exist_ok=True)
    oracle_targets = WORK / "gate1/dev/oracles/targets.jsonl"
    for grid, info in selected.items():
        b1_source = info["dir"] / "b1.jsonl"
        b2_source = info["dir"] / "b2.jsonl"
        b3_source = info["dir"] / "b3.jsonl"
        b4_source = info["dir"] / "b4_pool.jsonl"
        if any(
            not read_jsonl(path)
            for path in (b1_source, b2_source, b3_source, b4_source)
        ):
            continue
        b1_kept, _ = dedup_variant(grid, "b1", b1_source)
        b2_kept, _ = dedup_variant(grid, "b2", b2_source)
        b3_kept, b3_report = dedup_variant(grid, "b3", b3_source)
        b4_pool_kept, b4_pool_report = dedup_variant(grid, "b4_pool", b4_source)
        b4_kept = WORK / "gate2/dev/dedup" / f"{grid}-b4-matched.jsonl"
        run_cli(
            "match-control", "--config", str(CONFIG_PATH), "--cohort", "dev",
            "--b3", str(b3_kept), "--b4-pool", str(b4_pool_kept), "--output", str(b4_kept),
        )
        summary_payload = json.loads(
            (info["dir"] / "summary.json").read_text(encoding="utf-8")
        )
        try:
            equal_budget = validate_primary_equal_budget(
                base_keyframes=int(summary_payload["base_keyframes"]),
                cap_ratio=float(config.coverage.delta_cap_ratio),
                arm_rows={
                    "B1": read_jsonl(b1_kept),
                    "B2": read_jsonl(b2_kept),
                    "B3": read_jsonl(b3_kept),
                    "B4": read_jsonl(b4_kept),
                },
            )
        except RuntimeError as error:
            print(f"skip grid {grid}: primary equal-budget BLOCKED: {error}")
            continue
        budget_path = write_acceptance_json(
            WORK / "gate2/dev/equal-budget" / f"{grid}.json", equal_budget
        )
        store.publish(budget_path, f"gate2/dev/equal-budget/{grid}.json")

        snapshots = {
            name: WORK / "gate2/dev/snapshots" / f"{grid}-{name.lower()}.db"
            for name in ("B1", "B2", "B3", "B4")
        }
        kept_by_arm = {
            "B1": b1_kept,
            "B2": b2_kept,
            "B3": b3_kept,
            "B4": b4_kept,
        }
        for name, candidates in kept_by_arm.items():
            run_cli(
                "snapshot", "--config", str(CONFIG_PATH), "--cohort", "dev",
                "--base-db", str(BASE_DB), "--candidates", str(candidates),
                "--output", str(snapshots[name]),
            )
        run_cli(
            "evaluate-dev", "--config", str(CONFIG_PATH), "--base-relational", str(BASE_DB),
            "--b1-relational", str(snapshots["B1"]),
            "--b2-relational", str(snapshots["B2"]),
            "--b3-relational", str(snapshots["B3"]),
            "--b4-relational", str(snapshots["B4"]),
            "--b1-delta-root", str(union_delta), "--b2-delta-root", str(union_delta),
            "--b3-delta-root", str(union_delta), "--b4-delta-root", str(union_delta),
            "--b1", str(b1_kept), "--b2", str(b2_kept),
            "--b3", str(b3_kept), "--b4", str(b4_kept),
            "--oracle-targets", str(oracle_targets),
            "--selection-summary", str(info["dir"] / "summary.json"),
            "--b3-dedup-report", str(b3_report), "--b4-pool-dedup-report", str(b4_pool_report),
            "--b4-match-report", str(b4_kept.with_name(f"{b4_kept.stem}-report.json")),
            "--repeats", str(LATENCY_REPEATS),
        )
        report = WORK / "gate2/dev/report.json"
        saved = grid_reports_dir / f"{grid}.json"
        shutil.copyfile(report, saved)
        store.publish(saved, f"gate2/dev/grid-reports/{saved.name}")
        reports.append((grid, saved, b3_kept))

    passing = []
    for grid, report, b3_path in reports:
        payload = json.loads(report.read_text(encoding="utf-8"))
        if payload.get("gate", {}).get("go"):
            passing.append((payload["variants"]["B3"]["mean_score"], grid, report, b3_path))
    if not passing:
        checkpoint("gate2", status="NO_GO_DEV", reports=len(reports))
        print("NO_GO_DEV: không có grid nào đạt official DEV gate; giữ BTC-only.")
        return
    _, best_grid, best_report, best_b3 = max(passing, key=lambda item: (item[0], item[1]))
    best = selected[best_grid]
    run_cli(
        "freeze", "--config", str(CONFIG_PATH), "--dev-report", str(best_report),
        "--b3", str(best_b3), "--threshold", str(best["threshold"]),
        "--target-gap", str(best["gap"]), "--k-max", str(best["k_max"]),
    )
    freeze = WORK / "gate2/freeze.json"
    checkpoint("gate2", status="PASS", selected_grid=best_grid, freeze=str(freeze))
    print(f"DEV PASS — frozen policy: {best_grid}; held-out vẫn chưa mở.")


pilot_gate2()

## 8 — Chốt phiên và artifact hand-off

In [ ]:
run_cli("status", "--config", str(CONFIG_PATH))
state_path = WORK / "run-state.json"
if state_path.is_file():
    print(state_path.read_text(encoding="utf-8"))
print(f"Output transport: {type(transport).__name__}")
print(f"Experiment artifact root: {config.resolve(config.transport.local_root) if RUN_MODE == 'SMOKE' else store.paths.root}")
print("Không chạy held-out/full corpus trong notebook này nếu chưa có freeze + Lead review.")